# Machine Learning Model

## Data Context

### Imports

In [ ]:
import pandas as pd

%run "../utils/notebook_utils.py"
%run "../utils/feature_categories.py"

### Dataset Info

In [ ]:
TRAINING_DATA = "../data/Training.parquet"
TARGET = "status"
full_data_frame = pd.read_parquet(TRAINING_DATA, engine='pyarrow')
full_data_frame

### Dataset Statistics

In [ ]:
full_data_frame.describe()

In [ ]:
for title, columns in FEATURE_CATEGORIES.items():
    data_frame = generate_sub_data_frame(columns, add_status=False)
    frame_box_plot(
        data_frame, title
    )

### Boolean Attributes
These are attributes that indicate whether a specific feature is present or not present.

In [ ]:
BOOLEAN_ATTRIBUTES = set(
    full_data_frame.loc[:, (full_data_frame.nunique() == 2)]. columns
)

len(BOOLEAN_ATTRIBUTES)

### Non-Boolean Attributes
These attributes are much more diverse and complex in comparison to boolean attributes. This includes numerical values asides from 0 and 1 and URLs themselves.

In [ ]:
NON_BOOL_ATTRIBUTES = set(
    full_data_frame.loc[:, (full_data_frame.nunique() > 2)].columns
)

len(NON_BOOL_ATTRIBUTES)

## Removing Redundant Attributes

### Constant Features
These are features whose values are all the same across every URL and thereby serve no purpose in distinguishing between a phishing link and a legitimate link.

In [ ]:
REDUNDANT_FEATURES = set(
    full_data_frame.loc[:, (full_data_frame.nunique() == 1)].columns
)

save_redundant_features(REDUNDANT_FEATURES)

REDUNDANT_FEATURES

Below we remove these features to address the curse of dimensionality.

In [ ]:
RELEVANT_FEATURES = get_relevant_features(full_data_frame)

### Highly Correlated Features
These features can be identified visually via a correlation matrix.

In [ ]:
cleaned_data_frame_1 = generate_sub_data_frame(
    RELEVANT_FEATURES
)

frame_correlation_matrix(cleaned_data_frame_1, annotate=False)

#### Top 10 Highest Correlation Pairings

In [ ]:
get_top_correlation_pairs(cleaned_data_frame_1, 10)

#### Highly Correlated Pairs
Each pair represents a highly-correlated intersecting point in the correlation matrix.

In [ ]:
get_highly_correlated_pairs(cleaned_data_frame_1)

#### Redundant Feature Inference from Correlation
From each of the highly-correlated pairing of features shown above, we can infer the feature that is most redundant based on each feature's correlation value to determining the *status* of a URL.

In [ ]:
REDUNDANT_FEATURES.update(
    get_redundant_correlated_features(cleaned_data_frame_1)
)

save_redundant_features(REDUNDANT_FEATURES)

get_redundant_correlated_features(cleaned_data_frame_1)

In [ ]:
REDUNDANT_FEATURES.update(
    get_low_target_correlation_features(cleaned_data_frame_1)
)

save_redundant_features(REDUNDANT_FEATURES)

get_low_target_correlation_features(cleaned_data_frame_1)

In [ ]:
REDUNDANT_FEATURES.update(
    get_redundant_correlated_features(cleaned_data_frame_1)
)

save_redundant_features(REDUNDANT_FEATURES)

REDUNDANT_FEATURES

In [ ]:
RELEVANT_FEATURES = get_relevant_features(cleaned_data_frame_1)

len(RELEVANT_FEATURES)

## Clean Dataset Context

In [ ]:
cleaned_data_frame_2 = generate_sub_data_frame(
    RELEVANT_FEATURES
)

frame_correlation_matrix(cleaned_data_frame_2, annotate=False)

In [ ]:
get_top_correlation_pairs(cleaned_data_frame_2, 10)

In [ ]:
get_correlation_to_target(
    set(cleaned_data_frame_2.columns)
)